In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [2]:
from copy import deepcopy


from enviroment_bj import BlackjackEnvironment, BlackjackConfig, ObservationConfig, StartStateConfig
from loss import BellmanLossConfig, LossPhaseWeightConfig
from model.agents import DuelingRecurrentDoubleDQN, FeedForwardDoubleDQN
from training import (
    train_model,
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    DualEpsilonConfig,
    NStepConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
)
from model.encoder import BlackjackObservationEncoder, EncoderConfig
from model.agents import DuelingRecurrentDoubleDQN, AgentNetworkConfig
from copy import deepcopy

# ============================================================
# OBSERVATION
# Baseline para aprender a jugar bien
# ------------------------------------------------------------
# Mantengo información útil de la mano y del contexto inmediato,
# pero quito memoria histórica del shoe para que primero domine
# hard/soft/pairs/double/split.
# ============================================================

observation_config = ObservationConfig(
    profile="minimal_basic_strategy",
    obs_include_table_rules=True,
    obs_include_visible_rules_only=True,
    obs_include_hidden_rules=False,
    obs_include_decision_phase=True,
    obs_include_available_bet_multipliers=True,
    obs_current_hand_mode="basic_strategy",
    obs_include_other_player_hands=False,
    obs_include_current_bet=False,
    obs_include_betting_context=True,
    obs_include_hand_context=True,
    obs_include_insurance_context=True,
    obs_include_temporal_context=False,
    obs_include_hands_since_shuffle=False,
    obs_include_estimated_shoe_progress=False,
    obs_include_last_hand_outcome=False,
    obs_include_recent_actions=False,
    obs_recent_actions_window=5,
    obs_include_observed_cards_history=False,
    obs_observed_cards_mode="rank_counts",
    obs_recent_cards_window=20,
    obs_reset_history_on_shuffle=True,
    obs_include_exact_shoe_composition=False,
    obs_include_discard_summary=False,
    obs_include_n_decks=False,
    obs_include_shoe_penetration_rule=False,
)

# ============================================================
# START STATE
# ------------------------------------------------------------
# Para baseline de playing, fresh_shoe.
# Saltar despues a unknown_progress.
# ============================================================

start_state_config = StartStateConfig(
    mode="fresh_shoe",
    min_burned_rounds=0,
    max_burned_rounds=0,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=False,
)

# ============================================================
# TABLE CONFIG
# ------------------------------------------------------------
# Reglas cercanas a casino, pero:
# - solo 1x para desactivar aprendizaje de apuesta
# - sin exogenous_cards
# - sin surrender para no abrir otra rama todavía
# - sin six-card charlie para no desviar la policy base
# ============================================================

blackjack_config = BlackjackConfig(
    n_decks=8,
    shoe_penetration=0.75,
    use_cut_card=True,
    visible_shoe_change=True,

    exogenous_cards=False,
    simulate_exogenous_visible_cards=False,
    exogenous_visible_cards_mode="disabled",

    dealer_hits_soft_17=False,          # S17
    blackjack_payout=1.5,               # 3:2
    dealer_peeks_for_blackjack=True,

    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    double_split_aces_allowed=False,

    split_rule="same_value",
    max_hands_after_split=2,
    max_split_depth_per_hand=1,
    resplit_aces_allowed=False,
    hit_split_aces_allowed=False,

    surrender_allowed=False,
    insurance_allowed=True,
    six_card_charlie_enabled=False,

    base_bet=1.0,
    bet_multipliers=(1,),
    strict_shoe_validation=False,

    observation=observation_config,
    observation_mode=None,
    expose_shoe_composition=False,
)

# ============================================================
# ENVS
# ------------------------------------------------------------
# Un solo entorno. Aquí no queremos domain randomization aún.
# ============================================================

base_seed = 18

envs = [
    BlackjackEnvironment(
        config=deepcopy(blackjack_config),
        seed=base_seed,
        start_state=start_state_config,
    )
]

# ============================================================
# ENCODER CONFIG
# ------------------------------------------------------------
# Alineado con minimal_basic_strategy.
# ============================================================

encoder_config = EncoderConfig.for_profile("minimal_basic_strategy")

# ============================================================
# MODEL
# ------------------------------------------------------------
# Feedforward primero. Más estable para aprender playing base.
# Luego ya migraremos a recurrent.
# ============================================================

model_config = AgentNetworkConfig.for_architecture(
    architecture="feedforward",
    encoder_profile="minimal_basic_strategy",
    feedforward_hidden_dims=(256, 256, 128),
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    use_phase_adapters=False,
    use_module_gating=False,
)

model = FeedForwardDoubleDQN(config=model_config)

# ============================================================
# EPSILON
# ------------------------------------------------------------
# Betting casi irrelevante porque solo existe bet_1x.
# Playing con más exploración que la que estabas usando.
# ============================================================

dual_epsilon_config = DualEpsilonConfig(
    betting=EpsilonScheduleConfig(
        start=0.20,
        end=0.02,
        decay_steps=40_000,
        evaluation_epsilon=0.0,
    ),
    playing=EpsilonScheduleConfig(
        start=0.25,
        end=0.05,
        decay_steps=80_000,
        evaluation_epsilon=0.0,
    ),
)

# ============================================================
# LOSS
# ------------------------------------------------------------
# En esta fase quiero privilegiar playing, no betting.
# ============================================================

loss_config = BellmanLossConfig(
    gamma=0.99,
    loss_type="huber",
    validate_current_actions=True,
    validate_next_action_mask=True,
    allow_terminal_without_legal_next_action=True,
    phase_weights=LossPhaseWeightConfig(
        enabled=True,
        betting_weight=0.25,
        playing_weight=1.50,
    ),
)

# ============================================================
# N-STEP
# ------------------------------------------------------------
# Más señal de crédito que 1-step, sin irse al extremo.
# ============================================================

n_step_config = NStepConfig(
    enabled=True,
    n_steps=3,
)

# ============================================================
# REPLAY BUFFER
# ------------------------------------------------------------
# Feedforward: sequence_length no importa operacionalmente,
# pero dejo config consistente para despues LSTM o GRU.
# ============================================================

replay_buffer_config = ReplayBufferConfig(
    capacity=120_000,
    batch_size=128,
    warmup_size=12_000,
    sequence_length=8,
    min_sequence_length=2,
)

# ============================================================
# OPTIMIZATION
# ============================================================

optimization_config = OptimizationConfig(
    optimizer="adamw",
    learning_rate=3e-4,
    weight_decay=1e-5,
    scheduler="step",
    scheduler_step_size=20_000,
    scheduler_gamma=0.97,
    gradient_clipping=True,
    max_grad_norm=5.0,
)

# ============================================================
# TARGET UPDATE
# ------------------------------------------------------------
# Más estable que hard cada muy pocas actualizaciones.
# ============================================================

target_update_config = TargetUpdateConfig(
    mode="soft",
    hard_update_interval=1000,   # ignorado en soft
    soft_tau=0.005,
)

# ============================================================
# EVALUATION
# ============================================================

evaluation_config = EvaluationConfig(
    enabled=True,
    every_n_epochs=1,
    num_rounds=2500,
    max_decisions=25_000,
)

# ============================================================
# CHECKPOINTS
# ============================================================

checkpoint_config = CheckpointConfig(
    directory=r"training_checkpoints\baseline_playing_feedforward_v1",
    save_latest=True,
    save_best_eval=True,
    save_periodic=True,
    periodic_interval_updates=2500,
    best_metric_name="ev_per_1000_hands",
    maximize_best_metric=True,
)

# ============================================================
# PRINTS
# ============================================================

print_config = PrintConfig(
    enable=True,
    print_run_summary=True,
    print_warmup_interval=1000,
    print_update_interval=200,
    print_collection_interval=1000,
    print_epoch_header=True,
    print_epoch_summary=True,
    print_eval_summary=True,
    include_segment_details=False,
)

# ============================================================
# TRAINER
# ------------------------------------------------------------
# Más pasos por época y más tiempo total.
# ============================================================

trainer_config = TrainerConfig(
    total_epochs=60,
    env_steps_per_epoch=4500,
    train_frequency=4,
    updates_per_train_step=1,
    max_updates_per_epoch=None,
    device="cpu",   # usa "cpu" si no tienes GPU
    seed=18,
    reset_hidden_on_round_end=False,
    sequence_end_on_done=False,
    flush_partial_sequences_at_epoch_end=True,
    loss=loss_config,
)

# ============================================================
# PIPELINE
# ============================================================

pipeline_config = TrainingPipelineConfig(
    trainer=trainer_config,
    replay_buffer=replay_buffer_config,
    epsilon=dual_epsilon_config,
    n_step=n_step_config,
    optimization=optimization_config,
    target_update=target_update_config,
    evaluation=evaluation_config,
    checkpoints=checkpoint_config,
    prints=print_config,
)

# ============================================================
# TRAIN
# ============================================================

result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
    resume=False,
    resume_checkpoint_path=None,
)

BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=minimal_basic_strategy | obs=minimal_basic_strategy | start=fresh_shoe
  Runtime    : device=cpu | epochs=60 | envs=1 | steps/epoch=4500 | updates/epoch~=1125 | params=114,063
  Optim      : optimizer=adamw | lr=3.00e-04 | loss=huber | gamma=0.9900 | grad_clip=True(5.00)
  Replay     : warmup=12000 | capacity=120000 | batch=128 | seq_len=8 | min_seq_len=2
  Explore    : eps_bet=0.200->0.020 (decay 40000) | eps_play=0.250->0.050 (decay 80000) | target=soft | interval=1000 | tau=0.0050
  Extras     : n_step=True(3) | phase_loss_w=True (bet 0.25, play 1.50) | phase_adapters=False | module_gating=False
  Eval / CKPT: eval_rounds=2500 | eval_decisions=25000 | checkpoints=training_checkpoints\baseline_playing_feedforward_v1
  Table      : decks=8 | pen=0.75 | S17=True | payout=1.50 | double=any_two_cards | split=same_value | DAS=True

=== Epoch 1/60 ===
[Warmup] buffer 1001/12000
[Warmup] buffer 2001/12000
[Warmup] b